In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("catalog_name", "", "catalog_name")
dbutils.widgets.text("config_schema_name", "", "config_schema_name")
dbutils.widgets.text("source_schema_name", "", "source_schema_name")
dbutils.widgets.text("table_name", "", "table_name")

catalog_name = dbutils.widgets.get("catalog_name")
config_schema_name = dbutils.widgets.get("config_schema_name")
source_schema_name = dbutils.widgets.get("source_schema_name")
table_name = dbutils.widgets.get("table_name")

In [0]:
query = f"""
SELECT
    m.table_name AS table_name,
    r.rule_name,
    m.column_name AS column,
    r.rule_function AS function,
    m.criticality AS criticality,
    m.arguments AS arguments,
    r.rule_type,
    m.is_active
FROM {catalog_name}.{config_schema_name}.dqx_rule_mappings m
JOIN {catalog_name}.{config_schema_name}.dqx_rule_definitions r ON m.rule_id = r.rule_id
WHERE lower(m.table_name) = lower('{catalog_name}.{source_schema_name}.{table_name}')
"""

dqx_mapped_df = spark.sql(query).filter("is_active=true")
display(dqx_mapped_df)

if dqx_mapped_df.isEmpty():
    raise Exception("No active rules found for the table")

## Apply DQX Checks

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from databricks.sdk import WorkspaceClient

from databricks.labs.dqx.engine import DQEngine
from databricks.labs.dqx.rule import DQRowRule,DQDatasetRule
from databricks.labs.dqx import check_funcs
from databricks.labs.dqx.config import TableChecksStorageConfig
from databricks.labs.dqx.config import InputConfig, OutputConfig

In [0]:
# 4. Initialize Engine and Run
ws = WorkspaceClient()
engine = DQEngine(ws)

## config - check by metadata

In [0]:
checks_config = []
for row in dqx_mapped_df.collect():
    import json
    args = {}
    if row['arguments']:
        for k, v in row['arguments'].items():
            try:
                args[k] = json.loads(v)
            except Exception:
                args[k] = v
    check_dict = {
        "criticality": row['criticality'],
        "check": {
            "function": row['function'],
            "arguments": args
        }
    }
    checks_config.append(check_dict)

In [0]:
for i in (checks_config):
    print(i)

In [0]:
# end-to-end quality checking flow
engine.apply_checks_by_metadata_and_save_in_table(
    input_config=InputConfig(f"{catalog_name}.{source_schema_name}.{table_name}"),
    checks=checks_config,
    output_config=OutputConfig(f"{catalog_name}.{source_schema_name}_output.{table_name}", mode="overwrite"),
    quarantine_config=None
)

In [0]:
stats_query = f"""
SELECT
    '{catalog_name}.{source_schema_name}.{table_name}' AS table_name,
    _errors,
    _warnings,
    current_date() AS run_date
FROM {catalog_name}.{source_schema_name}_output.{table_name}
"""
stats_df = spark.sql(stats_query).filter("_errors IS NOT NULL OR _warnings IS NOT NULL")

stats_table = f"{catalog_name}.{config_schema_name}.dqx_run_stats"
stats_df.write\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .option("replaceWhere", f"run_date = current_date()")\
    .saveAsTable(stats_table)
print("Written to table successfully")

In [0]:
display(spark.table(f"{catalog_name}.{source_schema_name}_output.{table_name}"))
display(spark.table(f"{catalog_name}.{config_schema_name}.dqx_run_stats"))